# Phase 4: Feature Engineering

## Objective

The objective of this notebook is to transform the cleaned datasets into a machine learning-ready dataset for demand forecasting.

This phase includes:

- Loading cleaned datasets
- Creating Calendar Dataset
- Merging datasets
- Creating date-based features
- Creating lag features
- Creating rolling statistics
- Creating inventory-related features
- Saving the final feature engineered dataset

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [3]:
sales = pd.read_csv("../data/processed/sales_clean.csv")

sku = pd.read_csv("../data/processed/sku_clean.csv")

inventory = pd.read_csv("../data/processed/inventory_clean.csv")

print("Datasets Loaded Successfully")

Datasets Loaded Successfully


# Convert Date Columns

Convert date columns into datetime format for feature extraction.

In [4]:
sales["date"] = pd.to_datetime(sales["date"])

inventory["last_restock_date"] = pd.to_datetime(
    inventory["last_restock_date"]
)

# Create Calendar Dataset

Generate a calendar table from the sales dates.

In [5]:
calendar = pd.DataFrame()

calendar["date"] = pd.date_range(
    sales["date"].min(),
    sales["date"].max()
)

calendar.head()

,date
0,2022-01-01
1,2022-01-02
2,2022-01-03
3,2022-01-04
4,2022-01-05


In [6]:
calendar["year"] = calendar["date"].dt.year

calendar["month"] = calendar["date"].dt.month

calendar["month_name"] = calendar["date"].dt.month_name()

calendar["quarter"] = calendar["date"].dt.quarter

calendar["week"] = calendar["date"].dt.isocalendar().week

calendar["day"] = calendar["date"].dt.day

calendar["day_name"] = calendar["date"].dt.day_name()

calendar["day_of_week"] = calendar["date"].dt.dayofweek

calendar["is_weekend"] = (
    calendar["day_of_week"] >= 5
).astype(int)

calendar.head()

,date,year,month,month_name,quarter,week,day,day_name,day_of_week,is_weekend
0,2022-01-01,2022,1,January,1,52,1,Saturday,5,1
1,2022-01-02,2022,1,January,1,52,2,Sunday,6,1
2,2022-01-03,2022,1,January,1,1,3,Monday,0,0
3,2022-01-04,2022,1,January,1,1,4,Tuesday,1,0
4,2022-01-05,2022,1,January,1,1,5,Wednesday,2,0


# Save Calendar Dataset

In [7]:
calendar.to_csv(
    "../data/processed/calendar.csv",
    index=False
)

print("Calendar Dataset Saved Successfully")

Calendar Dataset Saved Successfully


# Merge Sales with SKU Master

In [8]:
sales_sku = sales.merge(
    sku,
    on="sku_id",
    how="left"
)

sales_sku.head()

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price_x,total_value,channel,discount_pct,promo_id,sku_name,category,subcategory,unit_price_y,cost_price,brand
0,2022-01-04,RCPT03769900,ST12,SKU02521,CUST04953,1,1790.10,1790.10,Mobile App,0.0,NaN,FamilyChoice Men's Wear 250ml,Apparel & Footwear,Men's Wear,1790.10,1418.71,FamilyChoice
1,2024-04-17,RCPT04532952,ST21,SKU00934,CUST01824,5,1037.02,2602.92,In-Store,49.8,PROMO019,ValueChoice Ready Meals 500ml,Frozen Foods,Ready Meals,1037.02,659.32,ValueChoice
2,2022-03-21,RCPT03856784,ST22,SKU02101,CUST04375,3,1016.14,3048.42,In-Store,0.0,NaN,GoldenHarvest Frozen Meat Medium,Frozen Foods,Frozen Meat,1016.14,647.81,GoldenHarvest
3,2024-10-11,RCPT04398786,ST09,SKU03292,CUST06378,2,164.03,328.06,In-Store,0.0,NaN,SoftTouch Pulses & Lentils Family Pack,Grocery,Pulses & Lentils,164.03,112.68,SoftTouch
4,2025-08-21,RCPT04574486,ST05,SKU00921,CUST00431,1,3363.82,3363.82,In-Store,0.0,NaN,SoftTouch Mobile Accessories Pack of 12,Electronics & Accessories,Mobile Accessories,3363.82,1875.04,SoftTouch


# Merge Inventory Information

In [9]:
merged = sales_sku.merge(
    inventory,
    on=["store_id", "sku_id"],
    how="left"
)

merged.head()

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price_x,total_value,channel,discount_pct,promo_id,sku_name,category,subcategory,unit_price_y,cost_price,brand,stock_on_hand,reorder_point,safety_stock,last_restock_date
0,2022-01-04,RCPT03769900,ST12,SKU02521,CUST04953,1,1790.10,1790.10,Mobile App,0.0,NaN,FamilyChoice Men's Wear 250ml,Apparel & Footwear,Men's Wear,1790.10,1418.71,FamilyChoice,NaN,NaN,NaN,NaT
1,2024-04-17,RCPT04532952,ST21,SKU00934,CUST01824,5,1037.02,2602.92,In-Store,49.8,PROMO019,ValueChoice Ready Meals 500ml,Frozen Foods,Ready Meals,1037.02,659.32,ValueChoice,NaN,NaN,NaN,NaT
2,2022-03-21,RCPT03856784,ST22,SKU02101,CUST04375,3,1016.14,3048.42,In-Store,0.0,NaN,GoldenHarvest Frozen Meat Medium,Frozen Foods,Frozen Meat,1016.14,647.81,GoldenHarvest,NaN,NaN,NaN,NaT
3,2024-10-11,RCPT04398786,ST09,SKU03292,CUST06378,2,164.03,328.06,In-Store,0.0,NaN,SoftTouch Pulses & Lentils Family Pack,Grocery,Pulses & Lentils,164.03,112.68,SoftTouch,196.0,85.0,34.0,2025-11-12
4,2025-08-21,RCPT04574486,ST05,SKU00921,CUST00431,1,3363.82,3363.82,In-Store,0.0,NaN,SoftTouch Mobile Accessories Pack of 12,Electronics & Accessories,Mobile Accessories,3363.82,1875.04,SoftTouch,NaN,NaN,NaN,NaT


# Merge Calendar Dataset

In [10]:
merged = merged.merge(
    calendar,
    on="date",
    how="left"
)

merged.head()

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price_x,total_value,channel,discount_pct,promo_id,sku_name,category,subcategory,unit_price_y,cost_price,brand,stock_on_hand,reorder_point,safety_stock,last_restock_date,year,month,month_name,quarter,week,day,day_name,day_of_week,is_weekend
0,2022-01-04,RCPT03769900,ST12,SKU02521,CUST04953,1,1790.10,1790.10,Mobile App,0.0,NaN,FamilyChoice Men's Wear 250ml,Apparel & Footwear,Men's Wear,1790.10,1418.71,FamilyChoice,NaN,NaN,NaN,NaT,2022,1,January,1,1,4,Tuesday,1,0
1,2024-04-17,RCPT04532952,ST21,SKU00934,CUST01824,5,1037.02,2602.92,In-Store,49.8,PROMO019,ValueChoice Ready Meals 500ml,Frozen Foods,Ready Meals,1037.02,659.32,ValueChoice,NaN,NaN,NaN,NaT,2024,4,April,2,16,17,Wednesday,2,0
2,2022-03-21,RCPT03856784,ST22,SKU02101,CUST04375,3,1016.14,3048.42,In-Store,0.0,NaN,GoldenHarvest Frozen Meat Medium,Frozen Foods,Frozen Meat,1016.14,647.81,GoldenHarvest,NaN,NaN,NaN,NaT,2022,3,March,1,12,21,Monday,0,0
3,2024-10-11,RCPT04398786,ST09,SKU03292,CUST06378,2,164.03,328.06,In-Store,0.0,NaN,SoftTouch Pulses & Lentils Family Pack,Grocery,Pulses & Lentils,164.03,112.68,SoftTouch,196.0,85.0,34.0,2025-11-12,2024,10,October,4,41,11,Friday,4,0
4,2025-08-21,RCPT04574486,ST05,SKU00921,CUST00431,1,3363.82,3363.82,In-Store,0.0,NaN,SoftTouch Mobile Accessories Pack of 12,Electronics & Accessories,Mobile Accessories,3363.82,1875.04,SoftTouch,NaN,NaN,NaN,NaT,2025,8,August,3,34,21,Thursday,3,0


# Create Lag Features

Lag features help forecasting models learn from previous sales.

In [11]:
merged = merged.sort_values(["sku_id", "date"])

merged["lag_1"] = (
    merged.groupby("sku_id")["quantity"]
    .shift(1)
)

merged["lag_7"] = (
    merged.groupby("sku_id")["quantity"]
    .shift(7)
)

merged["lag_30"] = (
    merged.groupby("sku_id")["quantity"]
    .shift(30)
)

# Rolling Features

Create moving averages for demand forecasting.

In [12]:
merged["rolling_7"] = (
    merged.groupby("sku_id")["quantity"]
    .transform(lambda x: x.rolling(7).mean())
)

merged["rolling_30"] = (
    merged.groupby("sku_id")["quantity"]
    .transform(lambda x: x.rolling(30).mean())
)

# Inventory Features

In [13]:
merged["stock_gap"] = (
    merged["stock_on_hand"]
    - merged["reorder_point"]
)

merged["stock_status"] = np.where(
    merged["stock_on_hand"] <= merged["reorder_point"],
    "Low Stock",
    "Normal"
)

# Handle Missing Values Generated by Lag Features

In [14]:
# Fill missing values in numeric columns
numeric_cols = merged.select_dtypes(include=["number"]).columns
merged[numeric_cols] = merged[numeric_cols].fillna(0)

# Fill missing values in object/string columns
object_cols = merged.select_dtypes(include=["object"]).columns
merged[object_cols] = merged[object_cols].fillna("Unknown")

print("Missing values handled successfully.")

Missing values handled successfully.


C:\Users\HITESH\AppData\Local\Temp\ipykernel_12924\2395052298.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = merged.select_dtypes(include=["object"]).columns


# Final Dataset Overview

In [15]:
merged.info()

<class 'pandas.DataFrame'>
Index: 100000 entries, 72442 to 91443
Data columns (total 37 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   date               100000 non-null  datetime64[us]
 1   receipt_id         100000 non-null  str           
 2   store_id           100000 non-null  str           
 3   sku_id             100000 non-null  str           
 4   customer_id        100000 non-null  str           
 5   quantity           100000 non-null  int64         
 6   unit_price_x       100000 non-null  float64       
 7   total_value        100000 non-null  float64       
 8   channel            100000 non-null  str           
 9   discount_pct       100000 non-null  float64       
 10  promo_id           100000 non-null  str           
 11  sku_name           100000 non-null  str           
 12  category           100000 non-null  str           
 13  subcategory        100000 non-null  str           
 14  u

In [16]:
# Rename sales unit price
merged.rename(columns={"unit_price_x": "unit_price"}, inplace=True)

# Drop duplicate unit price from SKU table
merged.drop(columns=["unit_price_y"], inplace=True)

print("Duplicate columns handled successfully.")

Duplicate columns handled successfully.


In [17]:
merged.info()

<class 'pandas.DataFrame'>
Index: 100000 entries, 72442 to 91443
Data columns (total 36 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   date               100000 non-null  datetime64[us]
 1   receipt_id         100000 non-null  str           
 2   store_id           100000 non-null  str           
 3   sku_id             100000 non-null  str           
 4   customer_id        100000 non-null  str           
 5   quantity           100000 non-null  int64         
 6   unit_price         100000 non-null  float64       
 7   total_value        100000 non-null  float64       
 8   channel            100000 non-null  str           
 9   discount_pct       100000 non-null  float64       
 10  promo_id           100000 non-null  str           
 11  sku_name           100000 non-null  str           
 12  category           100000 non-null  str           
 13  subcategory        100000 non-null  str           
 14  c

In [18]:
import pandas as pd

merged = pd.read_csv("../data/processed/merged_dataset.csv")

forecast = merged.copy()

forecast.to_csv(
    "../data/processed/forecast_dataset.csv",
    index=False
)

print("Forecast Dataset Saved Successfully")

Forecast Dataset Saved Successfully


# Phase Summary

Feature engineering completed successfully.

Completed tasks:

- Loaded cleaned datasets
- Generated calendar dataset
- Merged sales, SKU, inventory and calendar data
- Created calendar features
- Created lag features
- Created rolling average features
- Created inventory features
- Saved the final feature engineered dataset

The dataset is now ready for Demand Forecasting using machine learning models.

In [19]:
import os

print(os.listdir("../data/processed"))

['calendar.csv', 'forecast_dataset.csv', 'inventory_clean.csv', 'merged_dataset.csv', 'model_comparison.csv', 'predictions.csv', 'risk_scored_dataset.csv', 'sales_clean.csv', 'sku_clean.csv']


In [20]:
# ==========================================
# CREATE NEW MERGED DATASET 
# ==========================================

merged_sample = merged.sample(
    n=100000,
    random_state=42
)

merged_sample.to_csv(
    "../data/processed/merged_dataset.csv",
    index=False
)

print("="*60)
print(" New merged_dataset.csv created")
print("="*60)
print("Rows :", len(merged_sample))

 New merged_dataset.csv created
Rows : 100000
